<a href="https://colab.research.google.com/github/AiEmStylix/btc_prediction/blob/main/sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📈 Phân tích cảm xúc (Sentiment Analysis) dữ liệu Bitcoin Tweets với FinBERT
Notebook này tải trực tiếp dataset từ Hugging Face, làm sạch sơ bộ và sử dụng mô hình FinBERT để chấm điểm cảm xúc (Tích cực, Tiêu cực, Trung tính).



In [ ]:
%pip install pandas torch transformers

In [ ]:
%pip install datasets tqdm

In [ ]:
import pandas as pd
import torch
from transformers import pipeline
import time

# Kiểm tra xem máy có hỗ trợ GPU không để tăng tốc độ chạy
device = 0 if torch.cuda.is_available() else -1
device_name = "GPU" if device == 0 else "CPU"
print(f"⚙️ Đang sử dụng thiết bị: {device_name}")

⚙️ Đang sử dụng thiết bị: GPU


In [ ]:
print("🚀 Đang tải dữ liệu trực tiếp từ Hugging Face...")

# Đổi /blob/ thành /resolve/ để lấy raw file
url = "https://huggingface.co/datasets/AiEmStylix/btc_tweets/resolve/main/dataset_cleaned.csv"

# Đọc dữ liệu
df = pd.read_csv(url)

# Ép kiểu và xử lý giá trị rỗng
df['clean_text'] = df['clean_text'].fillna("").astype(str)

# Loại bỏ các dòng quá ngắn (chỉ có link hoặc tag, không có ý nghĩa phân tích)
df = df[df['clean_text'].str.len() > 3].reset_index(drop=True)

print(f"✅ Đã tải và làm sạch sơ bộ: {len(df)} dòng.")

# Hiển thị 5 dòng đầu tiên
df.head()

🚀 Đang tải dữ liệu trực tiếp từ Hugging Face...


/tmp/ipykernel_2144/3034154873.py:7: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url)


✅ Đã tải và làm sạch sơ bộ: 2293015 dòng.


,user_name,user_location,user_description,user_created,user_followers,user_friends,user_favourites,user_verified,date,text,hashtags,source,is_retweet,clean_text
0,DeSota Wilson,"Atlanta, GA","Biz Consultant, real estate, fintech, startups...",2009-04-26 20:05:09,8534.0,7605,4838,False,2026-02-10 23:59:04,Blue Ridge Bank shares halted by NYSE after #b...,['bitcoin'],Twitter Web App,False,Blue Ridge Bank shares halted by NYSE after #b...
1,Tdlmatias,"London, England","IM Academy : The best #forex, #SelfEducation, ...",2014-11-10 10:50:37,128.0,332,924,False,2026-02-10 23:54:48,"Guys evening, I have read this article about B...",NaN,Twitter Web App,False,"Guys evening, I have read this article about B..."
2,Crypto is the future,NaN,I will post a lot of buying signals for BTC tr...,2019-09-28 16:48:12,625.0,129,14,False,2026-02-10 23:54:33,$BTC A big chance in a billion! Price: \487264...,"['Bitcoin', 'FX', 'BTC', 'crypto']",dlvr.it,False,$BTC A big chance in a billion! Price: \487264...
3,Alex Kirchmaier 🇦🇹🇸🇪 #FactsSuperspreader,Europa,Co-founder @RENJERJerky | Forbes 30Under30 | I...,2016-02-03 13:15:55,1249.0,1472,10482,False,2026-02-10 23:54:06,This network is secured by 9 508 nodes as of t...,['BTC'],Twitter Web App,False,This network is secured by 9 508 nodes as of t...
4,ZerrBenz™ ⚔ ✪ 20732,"Bkk, Thailand",I'm a cat slave 🐱 Interested in Blockchain · T...,2010-01-12 07:00:04,742.0,716,2444,False,2026-02-10 23:53:30,💹 Trade #Crypto on #Binance \n\n📌 Enjoy #Cashb...,"['Crypto', 'Binance', 'Cashback']",Twitter Web App,False,💹 Trade #Crypto on #Binance 📌 Enjoy #Cashback ...


In [ ]:
from transformers import AutoTokenizer

print("🔍 Bắt đầu kiểm tra độ dài dữ liệu...")

# 1. Khởi tạo bộ Tokenizer chuẩn của FinBERT
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

# 2. Ước lượng nhanh bằng số lượng từ (tách bằng khoảng trắng)
# Việc này chạy cực nhanh nhờ Pandas
df['word_count'] = df['clean_text'].apply(lambda x: len(str(x).split()))
max_words = df['word_count'].max()
print(f"1️⃣ Dòng tweet chứa nhiều từ nhất có: {max_words} từ.")

# 3. Đo lường chính xác bằng Tokenizer (Chỉ đo top 1000 dòng dài nhất để tiết kiệm thời gian)
# Lấy ra 1000 dòng có word_count lớn nhất
top_longest_texts = df.nlargest(1000, 'word_count')['clean_text'].tolist()

print("⏳ Đang đo lường số lượng tokens thực tế...")
# Encode và đếm số lượng tokens, không sử dụng truncation để lấy độ dài thật
token_lengths = [len(tokenizer.encode(text, truncation=False)) for text in top_longest_texts]

exact_max_tokens = max(token_lengths)
print(f"2️⃣ Độ dài TOKENS lớn nhất thực tế (max_length): {exact_max_tokens} tokens.")

# 4. Gợi ý cấu hình tối ưu
recommended_max_length = min(exact_max_tokens + 5, 512) # Cộng thêm 5 làm biên an toàn, tối đa không quá 512
print(f"\n💡 GỢI Ý CÀI ĐẶT: Bạn nên set `max_length={recommended_max_length}` cho pipeline.")

🔍 Bắt đầu kiểm tra độ dài dữ liệu...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1️⃣ Dòng tweet chứa nhiều từ nhất có: 128 từ.
⏳ Đang đo lường số lượng tokens thực tế...
2️⃣ Độ dài TOKENS lớn nhất thực tế (max_length): 145 tokens.

💡 GỢI Ý CÀI ĐẶT: Bạn nên set `max_length=150` cho pipeline.


In [ ]:
import torch
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset # Công cụ sửa lỗi nạp dữ liệu
from datasets import Dataset
from tqdm.auto import tqdm
import time

print("🚀 Đang khởi tạo mô hình với các tối ưu hóa phần cứng...")

# 1. Khởi tạo Pipeline (Đã sửa torch_dtype thành dtype)
analyzer = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert",
    device=0,
    dtype=torch.float16, # Đã cập nhật theo chuẩn mới
    top_k=None
)

# Đảm bảo 100% dữ liệu là chuỗi (string) để tránh lỗi tokenizer
df['clean_text'] = df['clean_text'].fillna("").astype(str)

# 2. Chuyển Pandas DataFrame sang Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df[['clean_text']])

BATCH_SIZE = 2048

print(f"⚡ Bắt đầu xử lý với Batch Size: {BATCH_SIZE} | Data Type: FP16")
start_time = time.time()

results = []
# SỬ DỤNG KeyDataset ĐỂ SỬA LỖI VALUE_ERROR
for out in tqdm(analyzer(KeyDataset(hf_dataset, 'clean_text'), batch_size=BATCH_SIZE, truncation=True, max_length=156), total=len(hf_dataset)):
    results.append(out)

print("⏳ Đang trích xuất điểm số...")

labels, prob_pos, prob_neg, prob_neu = [], [], [], []

for res in results:
    labels.append(res[0]['label'])
    scores_dict = {item['label']: item['score'] for item in res}
    prob_pos.append(scores_dict.get('positive', 0.0))
    prob_neg.append(scores_dict.get('negative', 0.0))
    prob_neu.append(scores_dict.get('neutral', 0.0))

# Gắn kết quả vào DataFrame
df['sentiment'] = labels
df['prob_positive'] = prob_pos
df['prob_negative'] = prob_neg
df['prob_neutral'] = prob_neu
df['sentiment_score'] = df['prob_positive'] - df['prob_negative']

elapsed = time.time() - start_time
print(f"🎉 Hoàn thành siêu tốc sau {elapsed:.2f} giây!")

🚀 Đang khởi tạo mô hình với các tối ưu hóa phần cứng...


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

⚡ Bắt đầu xử lý với Batch Size: 2048 | Data Type: FP16


  0%|          | 0/2293015 [00:00<?, ?it/s]

⏳ Đang trích xuất điểm số...
🎉 Hoàn thành siêu tốc sau 5771.03 giây!


In [ ]:
output_file = "btc_tweets_MAX_SPEED.csv"

df.to_csv(output_file, index=False)

In [ ]:
from google.colab import drive
import shutil

# 1. Kết nối Google Drive (nếu bạn chưa kết nối trước đó)
drive.mount('/content/drive')

# 2. Định nghĩa tên file nguồn và đường dẫn đích trên Google Drive
source_file = "btc_tweets_MAX_SPEED.csv"
# Bạn có thể đổi tên thư mục đích nếu muốn lưu vào một folder cụ thể trong Drive
destination_file = "/content/drive/MyDrive/btc_tweets_MAX_SPEED.csv"

# 3. Copy file từ ổ cứng tạm thời của Colab sang Google Drive
shutil.copy(source_file, destination_file)

print(f"✅ Đã lưu file thành công lên Google Drive tại: {destination_file}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Đã lưu file thành công lên Google Drive tại: /content/drive/MyDrive/btc_tweets_MAX_SPEED.csv


In [ ]:
df_result = pd.read_csv(output_file)

# Hiển thị 5 dòng đầu tiên với các cột quan trọng nhất
display(df_result[['clean_text', 'sentiment', 'sentiment_score', 'prob_positive', 'prob_negative', 'prob_neutral']].head())

# In ra số lượng thống kê tổng quan của từng loại cảm xúc
print("\n📊 Thống kê số lượng tweet theo nhãn cảm xúc:")
print(df_result['sentiment'].value_type if hasattr(df_result['sentiment'], 'value_type') else df_result['sentiment'].value_counts())

/tmp/ipykernel_2144/3208250647.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_result = pd.read_csv(output_file)


,clean_text,sentiment,sentiment_score,prob_positive,prob_negative,prob_neutral
0,Blue Ridge Bank shares halted by NYSE after #b...,negative,-0.816891,0.012733,0.829624,0.157642
1,"Guys evening, I have read this article about B...",neutral,0.088455,0.099395,0.010940,0.889665
2,$BTC A big chance in a billion! Price: \487264...,neutral,0.019430,0.046343,0.026913,0.926744
3,This network is secured by 9 508 nodes as of t...,neutral,-0.027168,0.060511,0.087679,0.851810
4,💹 Trade #Crypto on #Binance 📌 Enjoy #Cashback ...,neutral,0.060822,0.071859,0.011037,0.917103



📊 Thống kê số lượng tweet theo nhãn cảm xúc:
sentiment
neutral     1994210
negative     170105
positive     128700
Name: count, dtype: int64
